### Model Tuning Procedure (Fixed Test + 5-Fold CV)

1. Build one fixed stratified `test.csv` (never used for tuning).
2. Put all remaining rows into `trainval.csv`.
3. Create 5 stratified folds from `trainval.csv` for tuning.
4. Tune with fold train/val files and average metrics across folds.
5. Retrain final model on all `trainval.csv` and evaluate once on `test.csv`.

In [1]:
from pathlib import Path
import importlib
import pandas as pd
import src.preprocess.split as split_module

# Reload to pick up latest edits when the notebook kernel already imported the module.
split_module = importlib.reload(split_module)
create_test_and_cv_splits = split_module.create_test_and_cv_splits

ANNOTATIONS_DIR = Path(r"C:\home\ben\REPSOL\Data\Annotations")
INPUT_CSV = ANNOTATIONS_DIR / "audio_annotations.csv"

# You can increase/decrease this depending on class sizes.
MIN_TEST_PER_CLASS = 10
N_SPLITS = 5

trainval_df, test_df, fold_summary_df, unmet_df = create_test_and_cv_splits(
    annotation_csv=INPUT_CSV,
    output_dir=ANNOTATIONS_DIR,
    seed=42,
    label_col="category",
    test_ratio=0.15,
    min_test_per_class=MIN_TEST_PER_CLASS,
    n_splits=N_SPLITS,
)

print("\nCreated files:")
for rel in [
    "trainval.csv",
    "test.csv",
    "cv_folds/fold_summary.csv",
]:
    p = ANNOTATIONS_DIR / rel
    print(rel, "->", "OK" if p.exists() else "MISSING")

print("\nTest set class counts:")
print(test_df["category"].value_counts().sort_index())

print("\nFold summary:")
print(fold_summary_df)

if not unmet_df.empty:
    print("\nCould not meet minimum test samples for some classes:")
    print(unmet_df)

# Optional sanity check: class distribution table
all_counts = pd.read_csv(INPUT_CSV)["category"].value_counts().sort_index().rename("all")
trainval_counts = trainval_df["category"].value_counts().sort_index().rename("trainval")
test_counts = test_df["category"].value_counts().sort_index().rename("test")

distribution = pd.concat([all_counts, trainval_counts, test_counts], axis=1).fillna(0).astype(int)
print("\nClass distribution (all/trainval/test):")
print(distribution)

Saved fixed test + CV folds to: C:\home\ben\REPSOL\Data\Annotations
Train+Val pool: 1672
Test: 304
CV folds: 5

Created files:
trainval.csv -> OK
test.csv -> OK
cv_folds/fold_summary.csv -> OK

Test set class counts:
category
0 ActividadBASE_NO pattern activity                                34
1 Pulses HAMMERING                                                 10
2 Marked cycles 3 segundos                                         20
3 Continuous activity & tone 3.15kHz_SPL ALTO                     106
4 Continuous activity & tone 3.15kHz_ SPL BAJO                     38
5 Blasts                                                           11
6 Machinery continuous activity                                    75
7 Works_ sirens and knocks en altas frecuencias RAFAGAS a 3.15     10
Name: count, dtype: int64

Fold summary:
   fold  train_size  val_size  min_class_count_in_val
0     1        1337       335                       6
1     2        1337       335                       7
2     3    

## Spectrogram Generation

Convert audio files to mel-spectrogram tensors (.pt files).
- Reads: audio from `Data/Audio/` using filenames in `trainval.csv` and `test.csv`
- Outputs: `Data/Spectrograms/trainval/` and `Data/Spectrograms/test/`

In [3]:
import subprocess
import sys

packages = ["torch", "librosa", "torchaudio", "scikit-learn", "pandas"]
for pkg in packages:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"✓ {pkg} already installed")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"✓ {pkg} installed")

Installing torch...
✓ torch installed
✓ librosa already installed
Installing torchaudio...
✓ torchaudio installed
Installing scikit-learn...
✓ scikit-learn installed
✓ pandas already installed


In [6]:
import importlib
import src.preprocess.create_spectrograms as spec_module

spec_module = importlib.reload(spec_module)

# Point to the actual audio location
AUDIO_DIR = Path(r"D:\REPSOL_Classification")

print(f"Audio directory: {AUDIO_DIR}")
print(f"Audio directory exists: {AUDIO_DIR.exists()}")

# Generate spectrograms for trainval and test from external audio source
spec_module.main(
    annotation_dir=ANNOTATIONS_DIR,
    audio_dir=AUDIO_DIR,
    spectrogram_dir=ANNOTATIONS_DIR.parent / "Spectrograms",
    splits=["trainval", "test"]
)

# Quick sanity check
import os
trainval_spec_dir = ANNOTATIONS_DIR.parent / "Spectrograms" / "trainval"
test_spec_dir = ANNOTATIONS_DIR.parent / "Spectrograms" / "test"

trainval_count = sum(1 for _ in trainval_spec_dir.rglob("*.pt")) if trainval_spec_dir.exists() else 0
test_count = sum(1 for _ in test_spec_dir.rglob("*.pt")) if test_spec_dir.exists() else 0

print(f"\nSpectrograms generated:")
print(f"  trainval: {trainval_count} .pt files")
print(f"  test: {test_count} .pt files")

Audio directory: D:\REPSOL_Classification
Audio directory exists: True

Processing trainval...
[1/1672] SM301165__0__20161001_013201_1.wav (0 ActividadBASE_NO pattern activity)
[2/1672] SM301165__0__20161001_013201_2.wav (0 ActividadBASE_NO pattern activity)
[3/1672] SM301165__0__20161001_013201_3.wav (0 ActividadBASE_NO pattern activity)
[4/1672] SM301165__0__20161001_051201_1.wav (0 ActividadBASE_NO pattern activity)
[5/1672] SM301165__0__20161001_051201_2.wav (0 ActividadBASE_NO pattern activity)
[6/1672] SM301165__0__20161001_051201_3.wav (0 ActividadBASE_NO pattern activity)
[7/1672] SM301165__0__20161001_053401_1.wav (0 ActividadBASE_NO pattern activity)
[8/1672] SM301165__0__20161001_053401_2.wav (0 ActividadBASE_NO pattern activity)
[9/1672] SM301165__0__20161001_053401_3.wav (0 ActividadBASE_NO pattern activity)
[10/1672] SM301165__0__20161001_055601_2.wav (0 ActividadBASE_NO pattern activity)
[11/1672] SM301165__0__20161001_055601_3.wav (0 ActividadBASE_NO pattern activity)
[

In [5]:
from pathlib import Path

SPEC_ROOT = ANNOTATIONS_DIR.parent / "Spectrograms"

trainval_pts = list((SPEC_ROOT / "trainval").rglob("*.pt")) if (SPEC_ROOT / "trainval").exists() else []
test_pts = list((SPEC_ROOT / "test").rglob("*.pt")) if (SPEC_ROOT / "test").exists() else []

print(f"\n✓ Spectrograms generated successfully!")
print(f"  trainval: {len(trainval_pts)} .pt files")
print(f"  test: {len(test_pts)} .pt files")
print(f"  Total: {len(trainval_pts) + len(test_pts)} .pt files")

# Show sample structure
print(f"\nSample trainval classes:")
trainval_classes = set()
for p in trainval_pts[:5]:
    trainval_classes.add(p.parent.name)
for cls in sorted(trainval_classes)[:3]:
    count = len(list((SPEC_ROOT / "trainval" / cls).glob("*.pt")))
    print(f"  {cls}: {count} files")


✓ Spectrograms generated successfully!
  trainval: 0 .pt files
  test: 0 .pt files
  Total: 0 .pt files

Sample trainval classes:
